# Fish Classification Model from Project PDF


In [ ]:
import os
import io
import base64
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image, ImageFilter, ImageEnhance
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import recall_score
from inference_sdk import InferenceHTTPClient
import matplotlib.pyplot as plt
from torchvision import transforms, models

# ========= Roboflow Setup ========
API_KEY = "YOUR_API_KEY"
INPUT_FOLDER = r"fish_dataset"
WORKSPACE = "shanwens-workspace"
WORKFLOW_ID = "fish-detection-cropped-1777939871146"

data_folder = INPUT_FOLDER

client = InferenceHTTPClient(
    api_url="https://serverless.roboflow.com",
    api_key=API_KEY
)

# ========== Detect Classes ===========
class_names = sorted([
    d for d in os.listdir(data_folder)
    if os.path.isdir(os.path.join(data_folder, d))
])

num_classes = len(class_names)
class_to_idx = {cls: i for i, cls in enumerate(class_names)}

# ================ Run Roboflow =============
cropped_images = []
cropped_labels = []

for class_name in class_names:
    input_folder = os.path.join(data_folder, class_name)
    label = class_to_idx[class_name]

    for filename in sorted(os.listdir(input_folder)):
        if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        filepath = os.path.join(input_folder, filename)

        result = client.run_workflow(
            workspace_name=WORKSPACE,
            workflow_id=WORKFLOW_ID,
            images={"image": filepath},
            use_cache=True
        )

        crop_list = result[0]["crops"]

        if len(crop_list) == 0:
            continue

        for b64_string in crop_list:
            img_bytes = base64.b64decode(b64_string)
            img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

            cropped_images.append(img)
            cropped_labels.append(label)

# ============= Data Augmentation ================
class RandomFishAugment:
    def __init__(self, input_size=(299,299)):
        self.input_size = input_size

    def __call__(self, img):
        img = img.resize(self.input_size)

        if random.random() < 0.4:
            k = random.randint(0,3)
            img = img.rotate(90*k)

        if random.random() < 0.4:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)

        if random.random() < 0.4:
            sigma = 3 * random.random()
            img = img.filter(ImageFilter.GaussianBlur(radius=sigma))

        if random.random() < 0.4:
            factor = 0.525 + (1.7 - 0.525) * random.random()
            img = ImageEnhance.Brightness(img).enhance(factor)

        return img

input_size = (299,299)

train_transform = transforms.Compose([
    RandomFishAugment(input_size),
    transforms.ToTensor()
])

val_test_transform = transforms.Compose([
    transforms.Resize(input_size),
    transforms.ToTensor()
])

# ======== Dataset ===========
class FishCropDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            img = self.transform(img)

        return img, label

# ======== Split Data =========
indices = np.arange(len(cropped_images))

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    stratify=cropped_labels,
    random_state=42
)

temp_labels = [cropped_labels[i] for i in temp_idx]

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=temp_labels,
    random_state=42
)

train_dataset_full = FishCropDataset(cropped_images, cropped_labels, transform=train_transform)
val_dataset_full = FishCropDataset(cropped_images, cropped_labels, transform=val_test_transform)
test_dataset_full = FishCropDataset(cropped_images, cropped_labels, transform=val_test_transform)

train_dataset = Subset(train_dataset_full, train_idx)
val_dataset = Subset(val_dataset_full, val_idx)
test_dataset = Subset(test_dataset_full, test_idx)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ============ Load ResNet18 ==============
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

num_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(num_features, num_classes)
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=5e-4
)

num_epochs = 20

# ============ Training ==============
for epoch in range(num_epochs):

    model.train()

    for images, labels_batch in train_loader:

        images = images.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels_batch)

        loss.backward()

        optimizer.step()

# ============ Testing ==============
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels_batch in test_loader:

        images = images.to(device)
        labels_batch = labels_batch.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels_batch.cpu().numpy())

accuracy = np.mean(np.array(all_preds) == np.array(all_labels))

print(f"Test Accuracy: {accuracy*100:.2f}%")

cm = confusion_matrix(all_labels, all_preds)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot()
plt.show()
